# Stage-2 XAttn-Only StreamVLM with Llama 3.2 3B Instruct

This notebook runs the frozen-backbone stage-2 alignment setup:
- frozen EfficientNet 3D-CNN stream encoder
- frozen `meta-llama/Llama-3.2-3B-Instruct`
- train only inserted cross-attention layers

Before running it on a cluster or MacBook Pro, make sure the environment has a recent `transformers` version compatible with Llama 3.2 and that you have access to the gated Meta checkpoint.

Notes:
- MPS requires a recent Apple Silicon PyTorch build.
- `flash-attn` and `bitsandbytes` are not used by this stage-2 path and should not be assumed available on Mac.

In [1]:
from pathlib import Path
import sys

repo_root = Path("/common/home/users/k/kristle.uy.2022/jupyterlab-venv-py-3117/updated_runnable")
if not (repo_root / "src").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(repo_root)
# repo_root = Path.cwd()

# if str(repo_root) not in sys.path:
#     sys.path.insert(0, str(repo_root))

# print(repo_root)

/common/home/users/k/kristle.uy.2022/jupyterlab-venv-py-3117/updated_runnable


In [ ]:
import sys
!{sys.executable} -m pip install huggingface_hub
from huggingface_hub import login
login(token="hf_XXXXXXXXXXXXXXXXXXXXXXXXX")

/common/home/users/k/kristle.uy.2022/jupyterlab-venv-pytorch-240/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import sys

!{sys.executable} -m pip install \
    accelerate==1.13.0 \
    huggingface_hub==1.8.0 \
    numpy==2.4.3 \
    opencv-python==4.13.0.92 \
    peft==0.18.1 \
    safetensors==0.7.0 \
    tokenizers==0.22.2 \
    torch==2.11.0 \
    torchaudio==2.11.0 \
    torchvision==0.26.0 \
    tqdm==4.67.3 \
    transformers==5.4.0

print("Done")



Done


In [12]:
# import json
# from pathlib import Path

# combined = Path(config["data_root"])
# short_clips = combined / "short_clips"

# # Get all videos actually present in short_clips/
# actual_videos = set(p.name for p in short_clips.iterdir())
# print(f"Videos in short_clips/: {len(actual_videos)}")

# # Load and filter labels to only matching videos
# with open(combined / "fine_grained_labels.json") as f:
#     all_labels = json.load(f)
# print(f"Labels before filter: {len(all_labels)}")

# filtered_labels = [
#     e for e in all_labels
#     if Path(e["video_path"]).name in actual_videos
# ]
# print(f"Labels after filter: {len(filtered_labels)}")

# # Overwrite fine_grained_labels.json with filtered version
# with open(combined / "fine_grained_labels.json", "w") as f:
#     json.dump(filtered_labels, f)
# print("fine_grained_labels.json updated — rerun the dataset cell now")

In [13]:
# import json
# from pathlib import Path

# combined = Path(config["data_root"])
# short_clips = combined / "short_clips"

# # Get all videos actually present in short_clips/
# actual_videos = set(p.name for p in short_clips.iterdir())
# print(f"Videos in short_clips/: {len(actual_videos)}")

# # Fix feedbacks_short_clips.json
# feedbacks_path = combined / "feedbacks_short_clips.json"
# if feedbacks_path.exists():
#     with open(feedbacks_path) as f:
#         feedbacks = json.load(f)
#     print(f"Feedbacks before filter: {len(feedbacks)}")
#     filtered_feedbacks = [
#         e for e in feedbacks
#         if Path(e["video_path"]).name in actual_videos
#     ]
#     print(f"Feedbacks after filter: {len(filtered_feedbacks)}")
#     with open(feedbacks_path, "w") as f:
#         json.dump(filtered_feedbacks, f)
#     print("feedbacks_short_clips.json updated")

# # Fix questions.json
# questions_path = combined / "questions.json"
# if questions_path.exists():
#     with open(questions_path) as f:
#         questions = json.load(f)
#     print(f"Questions before filter: {len(questions)}")
#     filtered_questions = [
#         e for e in questions
#         if Path(e["video_path"]).name in actual_videos
#     ]
#     print(f"Questions after filter: {len(filtered_questions)}")
#     with open(questions_path, "w") as f:
#         json.dump(filtered_questions, f)
#     print("questions.json updated")

# print("All JSONs filtered — rerun dataset cell now")

In [4]:
from dataclasses import asdict

import torch
from torch.utils.data import DataLoader, Subset

from src.stage2 import (
    FrozenEfficientNetStreamEncoder,
    Stage2QEVDFit300KDataset,
    Stage2TrainingConfig,
    XAttnConfig,
    build_xattn_only_streamvlm,
    prepare_generation_inputs,
    prepare_training_batch,
    resolve_runtime,
    stage2_collate,
    train_stage2,
)

PROFILE = "cluster_a40" #"mac_m3_max"  # or "cluster_a40"
subset_size = 30000
val_subset_size = 128
subset_seed = 469
run_tag = "subset30000_e2_validated_instruct"
validation_frequency_steps = 2000
previous_invalid_run_dir = repo_root / "outputs" / "stage2_xattn_llama32_instruct_mac_m3_max_subset20000_e2"

profile_overrides = {
    "mac_m3_max": {
        "preferred_device": "mps",
        "num_workers": 0,
        "micro_batch_size": 1,
    },
    "cluster_a40": {
        "preferred_device": "cuda",
        "num_workers": 2,
        "micro_batch_size": 1,
    },
}

runtime = resolve_runtime(preferred_device=profile_overrides[PROFILE]["preferred_device"])

config = {
    "profile": PROFILE,
    "runtime": runtime,
    "data_root": repo_root / "data" / "combined",
    "split": "train",
    "llm_model_name_or_path": "meta-llama/Llama-3.2-3B-Instruct", 
    "vision_checkpoint_path": repo_root / "ckpts_efficientnet" / "fitness_ally_hypermodel" / "efficientnet4Lite_1.8.3.checkpoint",
    "output_dir": repo_root / "outputs" / f"stage2_xattn_llama32_{PROFILE}_{run_tag}",
    "num_workers": profile_overrides[PROFILE]["num_workers"],
}

xattn_config = XAttnConfig(
    adapter_insert_layers=(6, 8, 10, 12, 14, 16, 18, 20, 22),
    xattn_block_size=1,
    num_of_xattn_heads=1,
    attn_dim=None,
)

training_config = Stage2TrainingConfig(
    learning_rate=5e-6,
    weight_decay=0.01,
    adam_beta1=0.9,
    adam_beta2=0.95,
    grad_clip_norm=1.0,
    epochs=2,
    effective_batch_size=32,
    micro_batch_size=profile_overrides[PROFILE]["micro_batch_size"],
    log_every=10,
    eval_every_steps=validation_frequency_steps,
)

print("profile:", PROFILE)
print("runtime:", runtime)
print("run tag:", run_tag)
print("subset size:", subset_size)
print("validation subset size:", val_subset_size)
print("subset seed:", subset_seed)
print("validation frequency steps:", validation_frequency_steps)
print("previous invalid run dir:", previous_invalid_run_dir)
print(asdict(training_config))

profile: cluster_a40
runtime: Stage2RuntimeConfig(device='cuda', llm_dtype=torch.bfloat16, vision_dtype=torch.float32, use_autocast=True)
run tag: subset30000_e2_validated_instruct
subset size: 30000
validation subset size: 128
subset seed: 469
validation frequency steps: 2000
previous invalid run dir: /common/home/users/k/kristle.uy.2022/jupyterlab-venv-py-3117/updated_runnable/outputs/stage2_xattn_llama32_instruct_mac_m3_max_subset20000_e2
{'learning_rate': 5e-06, 'weight_decay': 0.01, 'adam_beta1': 0.9, 'adam_beta2': 0.95, 'grad_clip_norm': 1.0, 'epochs': 2, 'effective_batch_size': 32, 'micro_batch_size': 1, 'max_new_tokens_eval': 128, 'log_every': 10, 'eval_every_steps': 2000}


## Native QEVD-FIT-300K layout

This notebook expects `data_root` to point at the native `combined/` folder:

```text
combined/
  short_clips/
  fine_grained_labels.json
  feedbacks_short_clips.json
  questions.json
```

In [5]:
dataset = Stage2QEVDFit300KDataset(
    data_root=config["data_root"],
    split=config["split"],
)
original_dataset_size = len(dataset)
subset_generator = torch.Generator().manual_seed(subset_seed)
all_indices = torch.randperm(original_dataset_size, generator=subset_generator).tolist()
subset_size = min(subset_size, original_dataset_size)
remaining_after_train = max(0, original_dataset_size - subset_size)
val_subset_size = min(val_subset_size, remaining_after_train)
train_indices = all_indices[:subset_size]
val_indices = all_indices[subset_size : subset_size + val_subset_size]
train_subset = Subset(dataset, train_indices)
val_subset = Subset(dataset, val_indices)
trained_sample_idx = train_indices[0]
trained_sample = dataset.samples[trained_sample_idx]
probe_sample_idx = val_indices[0] if val_indices else train_indices[0]
probe_sample = dataset.samples[probe_sample_idx]
train_dataloader = DataLoader(
    train_subset,
    batch_size=training_config.micro_batch_size,
    shuffle=True,
    num_workers=config["num_workers"],
    collate_fn=stage2_collate,
)
validation_dataloader = DataLoader(
    val_subset,
    batch_size=training_config.micro_batch_size,
    shuffle=False,
    num_workers=config["num_workers"],
    collate_fn=stage2_collate,
)
train_val_overlap = len(set(train_indices) & set(val_indices))

print("original dataset size:", original_dataset_size)
print("training subset size:", len(train_subset))
print("validation subset size:", len(val_subset))
print("training dataloader batches:", len(train_dataloader))
print("validation dataloader batches:", len(validation_dataloader))
print("train/val overlap:", train_val_overlap)
print("dataset stats (full dataset):", dataset.stats)
print("trained sample index:", trained_sample_idx)
print("trained sample id:", trained_sample.sample_id)
print("probe sample index:", probe_sample_idx)
print("probe sample id:", probe_sample.sample_id)
print("probe sample path:", probe_sample.video_path)
print(train_subset[0])

original dataset size: 75243
training subset size: 30000
validation subset size: 128
training dataloader batches: 30000
validation dataloader batches: 128
train/val overlap: 0
dataset stats (full dataset): {'feedback_missing_video_paths': 0, 'questions_missing_video_paths': 0, 'feedback_records_total': 2617, 'question_records_total': 30000}
trained sample index: 69680
trained sample id: 00260858.mp4:questions.high_level:1
probe sample index: 20961
probe sample id: 00085723.mp4:questions.high_level:0
probe sample path: /common/home/users/k/kristle.uy.2022/jupyterlab-venv-py-3117/updated_runnable/data/combined/short_clips/00085723.mp4
{'sample_id': '00260858.mp4:questions.high_level:1', 'video_path': '/common/home/users/k/kristle.uy.2022/jupyterlab-venv-py-3117/updated_runnable/data/combined/short_clips/00260858.mp4', 'task_type': 'qa', 'answer': 'The user is doing leg lifts. They are lifting their right leg.', 'question': "What exercise is the user doing? Describe how they're doing it."

In [ ]:
model = build_xattn_only_streamvlm(
    llm_model_name_or_path=config["llm_model_name_or_path"],
    device=config["runtime"].device,
    torch_dtype=config["runtime"].llm_dtype,
    allow_tokenizer_resize=False,  # first attempt; set True only if semantic token resolution fails
    xattn_config=xattn_config,
    hf_token="hf_XXXXXXXXXXXXXXXXXXXXXXXXX" #hf_XXXXXXXXXXXXXXXXXXXXXXXXX" # REDACTED 
)
vision_encoder = FrozenEfficientNetStreamEncoder(
    checkpoint_path=config["vision_checkpoint_path"],
    device=config["runtime"].device,
    torch_dtype=config["runtime"].vision_dtype,
)
initial_xattn_snapshot = {
    name: parameter.detach().cpu().clone()
    for name, parameter in model.named_trainable_parameters()
}

print("model class:", type(model.model))
print("config class:", type(model.model.config))
print("native xattn path:", getattr(model, "_uses_native_xattn", False))
print("model dtype:", next(model.model.parameters()).dtype)
print(model.special_token_strings)
print(model.special_token_ids)

Loading weights: 100%|██████████| 254/254 [00:00<00:00, 17546.50it/s]


model class: <class 'transformers.models.llama.modeling_llama.LlamaForCausalLM'>
config class: <class 'transformers.models.llama.configuration_llama.LlamaConfig'>
native xattn path: False
model dtype: torch.bfloat16
{'vision': '<|reserved_special_token_0|>', 'answer_begin': '<|reserved_special_token_1|>', 'answer_end': '<|reserved_special_token_2|>'}
{'vision': 128002, 'answer_begin': 128003, 'answer_end': 128005}


In [13]:
def audit_generation(model_to_eval, sample_to_eval, label):
    generation_inputs = prepare_generation_inputs(
        sample_to_eval,
        model=model_to_eval,
        vision_encoder=vision_encoder,
    )
    generated_ids = model_to_eval.generate_greedy(
        input_ids=generation_inputs["input_ids"],
        attention_mask=generation_inputs["attention_mask"],
        vision_feats=generation_inputs["vision_feats"],
        vision_xattn_mask=generation_inputs["vision_xattn_mask"],
        max_new_tokens=training_config.max_new_tokens_eval,
    )
    decoded = model_to_eval.tokenizer.decode(generated_ids[0], skip_special_tokens=False)
    print(f"{label} generation:")
    print(decoded)
    return decoded

model.zero_grad(set_to_none=True)
first_batch = next(iter(train_dataloader))
prepared = prepare_training_batch(first_batch, model=model, vision_encoder=vision_encoder)

print("runtime device:", config["runtime"].device)
print("runtime llm dtype:", config["runtime"].llm_dtype)
print("runtime vision dtype:", config["runtime"].vision_dtype)
for key in ["input_ids", "attention_mask", "vision_xattn_mask", "labels"]:
    print(key, prepared[key].shape)
print("vision feats:", prepared["vision_feats"]["feats"].shape)
label_tokens = prepared["labels"][0][prepared["labels"][0] != -100].tolist()
print("decoded supervised target:", model.tokenizer.decode(label_tokens, skip_special_tokens=False))

outputs = model(
    input_ids=prepared["input_ids"],
    attention_mask=prepared["attention_mask"],
    vision_feats=prepared["vision_feats"],
    vision_xattn_mask=prepared["vision_xattn_mask"],
    labels=prepared["labels"],
)
pretrain_loss = outputs.loss.detach()
pretrain_loss_isfinite = bool(torch.isfinite(pretrain_loss).item())
pretrain_logits_isfinite = bool(torch.isfinite(outputs.logits).all().item())
print("pre-train loss:", float(pretrain_loss))
print("pre-train finite loss:", pretrain_loss_isfinite)
print("pre-train finite logits:", pretrain_logits_isfinite)

outputs.loss.backward()
pretrain_grad_summary = []
for name, parameter in model.named_trainable_parameters():
    grad = parameter.grad
    if grad is None:
        continue
    grad_isfinite = bool(torch.isfinite(grad).all().item())
    max_abs = float(grad.abs().max().item()) if grad.numel() else 0.0
    pretrain_grad_summary.append((name, grad_isfinite, max_abs))
nonfinite_grad_names = [name for name, grad_isfinite, _ in pretrain_grad_summary if not grad_isfinite]
print("trainable parameter count:", len(initial_xattn_snapshot))
print("grad tensors checked:", len(pretrain_grad_summary))
print("non-finite grad tensors:", len(nonfinite_grad_names))
print(nonfinite_grad_names[:10])

model.zero_grad(set_to_none=True)
pretrain_generation_text = audit_generation(model, probe_sample, "pre-train probe")

runtime device: cuda
runtime llm dtype: torch.bfloat16
runtime vision dtype: torch.float32
input_ids torch.Size([1, 128])
attention_mask torch.Size([1, 128])
vision_xattn_mask torch.Size([1, 128])
labels torch.Size([1, 128])
vision feats: torch.Size([1, 24, 35, 1280])
decoded supervised target: They are doing a moving plank, but they are starting late. This could mean that they are starting the plank in a position that is already shifted or off-balance, or it could refer to the timing of when they are starting the exercise in relation to a prompt or routine.<|reserved_special_token_2|>
pre-train loss: 3.35448956489563
pre-train finite loss: True
pre-train finite logits: True
trainable parameter count: 45
grad tensors checked: 45
non-finite grad tensors: 0
[]
pre-train probe generation:
<|begin_of_text|>You are an expert fitness coaching AI who coaches users as they exercise. You observe them silently, assess their performance, and answer any questions they have.
<|reserved_special_toke

In [14]:
history = train_stage2(
    model=model,
    dataloader=train_dataloader,
    vision_encoder=vision_encoder,
    config=training_config,
    output_dir=config["output_dir"],
    validation_dataloader=validation_dataloader,
    probe_sample=probe_sample,
)
history

epoch 1/2:   7%|▋         | 2001/30000 [05:57<35:52:15,  4.61s/it, loss=4.1046]

[eval] step=2000 train_loss=4.1046 val_loss=3.4722
[eval] probe generation:
<|begin_of_text|>You are an expert fitness coaching AI who coaches users as they exercise. You observe them silently, assess their performance, and answer any questions they have.
<|reserved_special_token_0|><|reserved_special_token_0|><|reserved_special_token_0|><|reserved_special_token_0|><|reserved_special_token_0|><|reserved_special_token_0|><|reserved_special_token_0|><|reserved_special_token_0|><|reserved_special_token_0|><|reserved_special_token_0|><|reserved_special_token_0|><|reserved_special_token_0|><|reserved_special_token_0|><|reserved_special_token_0|><|reserved_special_token_0|><|reserved_special_token_0|><|reserved_special_token_0|><|reserved_special_token_0|><|reserved_special_token_0|><|reserved_special_token_0|>
What is the user doing?
<|reserved_special_token_1|>The user is currently running on a treadmill. They are jogging at a moderate pace, with their feet pounding the belt in a steady rh

epoch 1/2:  13%|█▎        | 4001/30000 [11:55<33:09:24,  4.59s/it, loss=3.6297]

[eval] step=4000 train_loss=3.6297 val_loss=2.9115
[eval] probe generation:
<|begin_of_text|>You are an expert fitness coaching AI who coaches users as they exercise. You observe them silently, assess their performance, and answer any questions they have.
<|reserved_special_token_0|><|reserved_special_token_0|><|reserved_special_token_0|><|reserved_special_token_0|><|reserved_special_token_0|><|reserved_special_token_0|><|reserved_special_token_0|><|reserved_special_token_0|><|reserved_special_token_0|><|reserved_special_token_0|><|reserved_special_token_0|><|reserved_special_token_0|><|reserved_special_token_0|><|reserved_special_token_0|><|reserved_special_token_0|><|reserved_special_token_0|><|reserved_special_token_0|><|reserved_special_token_0|><|reserved_special_token_0|><|reserved_special_token_0|>
What is the user doing?
<|reserved_special_token_1|>The user is doing a push-up. The user is in a plank position, with their hands shoulder-width apart and their feet hip-width apart.

epoch 1/2:  20%|██        | 6000/30000 [17:52<43:17:51,  6.49s/it, loss=3.3338]

...

epoch 2/2:  93%|█████████▎| 28001/30000 [1:21:26<2:26:27,  4.40s/it, loss=1.8376]

[eval] step=58000 train_loss=1.8376 val_loss=1.8666
[eval] probe generation:
<|begin_of_text|>You are an expert fitness coaching AI who coaches users as they exercise. You observe them silently, assess their performance, and answer any questions they have.
<|reserved_special_token_0|><|reserved_special_token_0|><|reserved_special_token_0|><|reserved_special_token_0|><|reserved_special_token_0|><|reserved_special_token_0|><|reserved_special_token_0|><|reserved_special_token_0|><|reserved_special_token_0|><|reserved_special_token_0|><|reserved_special_token_0|><|reserved_special_token_0|><|reserved_special_token_0|><|reserved_special_token_0|><|reserved_special_token_0|><|reserved_special_token_0|><|reserved_special_token_0|><|reserved_special_token_0|><|reserved_special_token_0|><|reserved_special_token_0|>
What is the user doing?
<|reserved_special_token_1|>The user is doing a right arm wave with their right hand in the air while standing in a squat position with their left leg in fron

epoch 2/2: 100%|██████████| 30000/30000 [1:27:06<00:00,  5.74it/s, loss=1.8342]  


[eval] step=60000 train_loss=1.8342 val_loss=1.8598
[eval] probe generation:
<|begin_of_text|>You are an expert fitness coaching AI who coaches users as they exercise. You observe them silently, assess their performance, and answer any questions they have.
<|reserved_special_token_0|><|reserved_special_token_0|><|reserved_special_token_0|><|reserved_special_token_0|><|reserved_special_token_0|><|reserved_special_token_0|><|reserved_special_token_0|><|reserved_special_token_0|><|reserved_special_token_0|><|reserved_special_token_0|><|reserved_special_token_0|><|reserved_special_token_0|><|reserved_special_token_0|><|reserved_special_token_0|><|reserved_special_token_0|><|reserved_special_token_0|><|reserved_special_token_0|><|reserved_special_token_0|><|reserved_special_token_0|><|reserved_special_token_0|>
What is the user doing?
<|reserved_special_token_1|>The user is holding their breath while walking in a straight line.
<|python_tag|>The user is walking in a straight line while hold

[{'epoch': 1.0, 'avg_loss': 2.364489193077882, 'optimizer_steps': 938.0},
 {'epoch': 2.0, 'avg_loss': 1.8341841824690501, 'optimizer_steps': 1876.0}]

In [15]:
posttrain_batch = next(iter(train_dataloader))
posttrain_prepared = prepare_training_batch(posttrain_batch, model=model, vision_encoder=vision_encoder)

with torch.no_grad():
    posttrain_outputs = model(
        input_ids=posttrain_prepared["input_ids"],
        attention_mask=posttrain_prepared["attention_mask"],
        vision_feats=posttrain_prepared["vision_feats"],
        vision_xattn_mask=posttrain_prepared["vision_xattn_mask"],
        labels=posttrain_prepared["labels"],
    )

posttrain_loss = posttrain_outputs.loss.detach()
posttrain_loss_isfinite = bool(torch.isfinite(posttrain_loss).item())
posttrain_logits_isfinite = bool(torch.isfinite(posttrain_outputs.logits).all().item())
print("post-train loss:", float(posttrain_loss))
print("post-train finite loss:", posttrain_loss_isfinite)
print("post-train finite logits:", posttrain_logits_isfinite)
posttrain_generation_text = audit_generation(model, probe_sample, "post-train probe")

post-train loss: 0.7984379529953003
post-train finite loss: True
post-train finite logits: True
post-train probe generation:
<|begin_of_text|>You are an expert fitness coaching AI who coaches users as they exercise. You observe them silently, assess their performance, and answer any questions they have.
<|reserved_special_token_0|><|reserved_special_token_0|><|reserved_special_token_0|><|reserved_special_token_0|><|reserved_special_token_0|><|reserved_special_token_0|><|reserved_special_token_0|><|reserved_special_token_0|><|reserved_special_token_0|><|reserved_special_token_0|><|reserved_special_token_0|><|reserved_special_token_0|><|reserved_special_token_0|><|reserved_special_token_0|><|reserved_special_token_0|><|reserved_special_token_0|><|reserved_special_token_0|><|reserved_special_token_0|><|reserved_special_token_0|><|reserved_special_token_0|>
What is the user doing?
<|reserved_special_token_1|>The user is holding their breath while walking in a straight line.
<|python_tag|>T

In [16]:
import json

model.save_stage2_checkpoint(
    config["output_dir"] / "final",
    extra_config={
        "training_config": asdict(training_config),
        "llm_model_name_or_path": config["llm_model_name_or_path"],
        "vision_checkpoint_path": str(config["vision_checkpoint_path"]),
    },
)
print("audited run dir:", config["output_dir"])
print("previous invalid run dir exists:", previous_invalid_run_dir.exists())

checkpoint_dirs = [
    config["output_dir"] / "epoch_1",
    config["output_dir"] / "epoch_2",
    config["output_dir"] / "final",
]
checkpoint_audit = []
final_checkpoint_state = None

for checkpoint_dir in checkpoint_dirs:
    record = {"checkpoint": checkpoint_dir.name, "exists": checkpoint_dir.exists()}
    if checkpoint_dir.exists():
        checkpoint_config = json.loads((checkpoint_dir / "stage2_config.json").read_text())
        state_dict = torch.load(checkpoint_dir / "xattn_state_dict.pt", map_location="cpu")
        nonfinite_tensors = [
            name for name, tensor in state_dict.items()
            if not torch.isfinite(tensor).all().item()
        ]
        max_abs_delta_vs_init = 0.0
        changed_tensor_count = 0
        for name, tensor in state_dict.items():
            init_tensor = initial_xattn_snapshot.get(name)
            if init_tensor is None:
                continue
            delta = float((tensor - init_tensor).abs().max().item())
            if delta > 0.0:
                changed_tensor_count += 1
            if delta > max_abs_delta_vs_init:
                max_abs_delta_vs_init = delta
        if checkpoint_dir.name == "final":
            final_checkpoint_state = state_dict

        record.update({
            "training_config": checkpoint_config.get("extra_config", {}).get("training_config"),
            "nonfinite_tensor_count": len(nonfinite_tensors),
            "nonfinite_tensors_preview": nonfinite_tensors[:10],
            "changed_tensor_count_vs_init": changed_tensor_count,
            "max_abs_delta_vs_init": max_abs_delta_vs_init,
        })
    checkpoint_audit.append(record)

checkpoint_audit

audited run dir: /common/home/users/k/kristle.uy.2022/jupyterlab-venv-py-3117/updated_runnable/outputs/stage2_xattn_llama32_cluster_a40_subset30000_e2_validated_instruct
previous invalid run dir exists: False


[{'checkpoint': 'epoch_1',
  'exists': True,
  'training_config': {'adam_beta1': 0.9,
   'adam_beta2': 0.95,
   'effective_batch_size': 32,
   'epochs': 2,
   'eval_every_steps': 2000,
   'grad_clip_norm': 1.0,
   'learning_rate': 5e-06,
   'log_every': 10,
   'max_new_tokens_eval': 128,
   'micro_batch_size': 1,
   'weight_decay': 0.01},
  'nonfinite_tensor_count': 0,
  'nonfinite_tensors_preview': [],
  'changed_tensor_count_vs_init': 45,
  'max_abs_delta_vs_init': 0.00390625},
 {'checkpoint': 'epoch_2',
  'exists': True,
  'training_config': {'adam_beta1': 0.9,
   'adam_beta2': 0.95,
   'effective_batch_size': 32,
   'epochs': 2,
   'eval_every_steps': 2000,
   'grad_clip_norm': 1.0,
   'learning_rate': 5e-06,
   'log_every': 10,
   'max_new_tokens_eval': 128,
   'micro_batch_size': 1,
   'weight_decay': 0.01},
  'nonfinite_tensor_count': 0,
  'nonfinite_tensors_preview': [],
  'changed_tensor_count_vs_init': 45,
  'max_abs_delta_vs_init': 0.00390625},
 {'checkpoint': 'final',
  'ex

In [17]:
import math

current_nonfinite_params = [
    name for name, parameter in model.named_trainable_parameters()
    if not torch.isfinite(parameter.detach()).all().item()
]
current_vs_final_max_delta = None
if final_checkpoint_state is not None:
    current_vs_final_max_delta = max(
        float((parameter.detach().cpu() - final_checkpoint_state[name]).abs().max().item())
        for name, parameter in model.named_trainable_parameters()
        if name in final_checkpoint_state
    )

def answer_segment(decoded_text):
    answer_begin = model.special_token_strings["answer_begin"]
    answer_end = model.special_token_strings["answer_end"]
    if answer_begin in decoded_text:
        decoded_text = decoded_text.split(answer_begin, 1)[1]
    if answer_end in decoded_text:
        decoded_text = decoded_text.split(answer_end, 1)[0]
    return decoded_text.strip()

def is_degenerate_answer(text):
    answer = answer_segment(text)
    if not answer:
        return True
    return answer.count("!") >= 16 or len(set(answer)) <= 3

history_all_nan = all(math.isnan(record["avg_loss"]) for record in history)
checkpoint_nonfinite = any(record.get("nonfinite_tensor_count", 0) > 0 for record in checkpoint_audit)

if not pretrain_loss_isfinite or not pretrain_logits_isfinite:
    audit_verdict = "Invalid run: model compatibility/integration"
elif checkpoint_nonfinite or history_all_nan or (not posttrain_loss_isfinite) or nonfinite_grad_names or current_nonfinite_params:
    audit_verdict = "Invalid run: numeric instability"
elif is_degenerate_answer(posttrain_generation_text):
    audit_verdict = "Valid but weak run"
else:
    audit_verdict = "Valid but weak run"

audit_summary = {
    "run_dir": str(config["output_dir"]),
    "subset_size": subset_size,
    "val_subset_size": val_subset_size,
    "epochs": training_config.epochs,
    "device": config["runtime"].device,
    "llm_dtype": str(config["runtime"].llm_dtype),
    "vision_dtype": str(config["runtime"].vision_dtype),
    "history_all_nan": history_all_nan,
    "pretrain_loss_isfinite": pretrain_loss_isfinite,
    "pretrain_logits_isfinite": pretrain_logits_isfinite,
    "nonfinite_grad_tensors": len(nonfinite_grad_names),
    "posttrain_loss_isfinite": posttrain_loss_isfinite,
    "posttrain_logits_isfinite": posttrain_logits_isfinite,
    "current_nonfinite_params": len(current_nonfinite_params),
    "current_vs_final_max_delta": current_vs_final_max_delta,
    "posttrain_answer_preview": answer_segment(posttrain_generation_text)[:200],
    "verdict": audit_verdict,
}
audit_summary

{'run_dir': '/common/home/users/k/kristle.uy.2022/jupyterlab-venv-py-3117/updated_runnable/outputs/stage2_xattn_llama32_cluster_a40_subset30000_e2_validated_instruct',
 'subset_size': 30000,
 'val_subset_size': 128,
 'epochs': 2,
 'device': 'cuda',
 'llm_dtype': 'torch.bfloat16',
 'vision_dtype': 'torch.float32',
 'history_all_nan': False,
 'pretrain_loss_isfinite': True,
 'pretrain_logits_isfinite': True,
 'nonfinite_grad_tensors': 0,
 'posttrain_loss_isfinite': True,
 'posttrain_logits_isfinite': True,
 'current_nonfinite_params': 0,
 'current_vs_final_max_delta': 0.0,
 'posttrain_answer_preview': 'The user is holding their breath while walking in a straight line.\n<|python_tag|>The user is walking in a straight line while holding their breath.<|eot_id|><|start_header_id|><|python_tag|>The user i',
 'verdict': 'Valid but weak run'}